
# Работа с данными бизнеса в PySpark

## Шаг 1. Загрузка данных и знакомство с ними

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum, dayofweek, to_date, collect_set
from pyspark.sql.types import IntegerType

# Создаём Spark-сессию
spark = SparkSession.builder \
    .appName("Create DataFrame and Read from S3") \
    .config("fs.s3a.endpoint", "storage.yandexcloud.net") \
    .getOrCreate()

In [4]:
# Загружаем и переименовываем столбцы для audition
audition_df = (
    spark.read
        .option("header", "false")
        .option("inferSchema", "true")
        .csv("audition.csv")
)

# Выводим 10 строк и схему оригинала
print("Первые 10 строк оригинального:")
audition_df.show(10)

print("\nСхема данных:")
audition_df.printSchema()

# Переименовываем столбцы и удаляем лишний c0
audition_df = audition_df.select(
    col("_c1").alias("audition_id"),
    col("_c2").alias("puid"),
    col("_c3").alias("usage_platform_ru"),
    col("_c4").alias("msk_business_dt_str"),
    col("_c5").alias("app_version"),
    col("_c6").alias("adult_content_flg"),
    col("_c7").alias("hours"),
    col("_c8").alias("hours_sessions_long"),
    col("_c9").alias("kids_content_flg"),
    col("_c10").alias("main_content_id"),
    col("_c11").alias("usage_geo_id_name"),
    col("_c12").alias("usage_country_name")
)

# Выводим 10 строк и схему испрвленую под описание схему из описания данных
print("Первые 10 строк отредактированного:")
audition_df.show(10)

print("\nСхема данных:")
audition_df.printSchema()

Первые 10 строк оригинального:


+-----+---+--------------------+---------------+----------+----------+-----+------------------+------------------+-----+--------+--------------------+---------+
|  _c0|_c1|                 _c2|            _c3|       _c4|       _c5|  _c6|               _c7|               _c8|  _c9|    _c10|                _c11|     _c12|
+-----+---+--------------------+---------------+----------+----------+-----+------------------+------------------+-----+--------+--------------------+---------+
|  162|  0|68296628-f9d6-11e...|        Станция|2024-11-26|      null|false|0.0377777777777777|0.0377777777777777| true|oCURrBKV|              Алматы|Казахстан|
|  213|  1|682966dc-f9d6-11e...|        Станция|2024-11-26|      null|false| 8.333333333333E-4|               0.0| true|qOL0JJL5|              Москва|   Россия|
|   63|  2|682966dc-f9d6-11e...|        Станция|2024-11-26|      null|false|0.0044444444444444|               0.0| true|ndM5nzgT|             Иркутск|   Россия|
|    2|  4|68296704-f9d6-11e...|  

+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+------------------+
|audition_id|                puid|usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|             hours|hours_sessions_long|kids_content_flg|main_content_id|   usage_geo_id_name|usage_country_name|
+-----------+--------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+--------------------+------------------+
|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|       null|            false|0.0377777777777777| 0.0377777777777777|            true|       oCURrBKV|              Алматы|         Казахстан|
|          1|682966dc-f9d6-11e...|          Станция|         2024-11-26|       null|            false| 8.333333333333E-4|                0.0

In [3]:
# Загружаем и переименовываем столбцы для content
content_df = (
    spark.read
        .option("header", "false")
        .option("inferSchema", "true")
        .csv("content.csv")
)

# Выводим 10 строк и схему оригинала
print("Первые 10 строк оригинала:")
content_df.show(10)

print("\nСхема данных:")
content_df.printSchema()

content_df = content_df.select(
    col("_c0").alias("main_content_id"),
    col("_c1").alias("main_content_type"),
    col("_c2").alias("main_content_name"),
    col("_c3").alias("main_content_duration_hours"),
    col("_c4").alias("published_topic_title_list"),
    col("_c5").alias("main_author_name")
)

# Выводим 10 строк и схему испрвленую под описание схему из описания данных
print("Первые 10 строк отредактированного:")
content_df.show(10)

print("\nСхема данных:")
content_df.printSchema()

Первые 10 строк:
+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|main_content_id|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|    main_author_name|
+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|       A00hxVIL|             Book|Давай поговорим о...|                      4.151|      'Синхронизировано...|        Карл Ричардс|
|       A03if3j5|        Comicbook|Минимализм из ком...|                  2.6854546|      'Уборка и организ...|Элизабет Энрайт Ф...|
|       A05XPJgr|             Book| Гоните ваши денежки|                  6.9114366|      'Художественная л...|Наталья Александрова|
|       A06fBP22|             Book|Крайон. Судьбу мо...|                  3.5218182|      'Эзотерика', 'Окк...|        Тамара Шмидт|
|       A0FkgwIl|        Comicbook|Майор Гром. Допол

In [5]:
# Проверяем объем данных
print(f"Количество строк в audition_df: {audition_df.count()}")
print(f"\nКоличество строк в content_df: {content_df.count()}")


Количество строк в audition_df: 1002896
Количество строк в content_df: 31668


**Вывод по шагу 1:**
- Провели загрузку данных
- Убедились в правильности типов данных у столбцов
- Обработали таблицы под таблицы из описания данных(дали название колонок, убрали лишнее)
- Проверили обьем данных обеих таблиц(размеры различаются так как в одной таблице хранятся пользователи а в другой контент платформы что логично для большого сервиса с большим количеством пользователей)

## Шаг 2. Трансформация и преобразование таблиц

In [6]:
# Создаем колонку с длительностью сессий в минутах
long_sessions_minutes_df = audition_df.select(
    col("puid"),
    col("hours_sessions_long")
).withColumn(
    "minutes_sessions_long", 
    (col("hours_sessions_long") * 60).cast(IntegerType())
)

print("Длительность сессий в минутах (первые 10 строк):")
long_sessions_minutes_df.show(10)


Длительность сессий в минутах (первые 10 строк):


+--------------------+-------------------+---------------------+
|                puid|hours_sessions_long|minutes_sessions_long|
+--------------------+-------------------+---------------------+
|68296628-f9d6-11e...| 0.0377777777777777|                    2|
|682966dc-f9d6-11e...|                0.0|                    0|
|682966dc-f9d6-11e...|                0.0|                    0|
|68296704-f9d6-11e...|                0.0|                    0|
|68296722-f9d6-11e...|                0.0|                    0|
|68296740-f9d6-11e...| 0.4890794444444444|                   29|
|68296740-f9d6-11e...|          0.1847275|                   11|
|6829675e-f9d6-11e...| 1.4530555555555555|                   87|
|6829677c-f9d6-11e...| 0.5399818181818182|                   32|
|6829679a-f9d6-11e...|                0.0|                    0|
+--------------------+-------------------+---------------------+
only showing top 10 rows



In [7]:
# Добавляем информацию о выходных днях
sessions_with_weekend_df = audition_df.select(
    col("puid"),
    col("hours_sessions_long"),
    col("msk_business_dt_str")
).withColumn(
    "minutes_sessions_long", 
    (col("hours_sessions_long") * 60).cast(IntegerType())
).withColumn(
    "is_weekend",
    when(dayofweek(to_date(col("msk_business_dt_str"), "yyyy-MM-dd")).isin([1, 7]), True).otherwise(False)
)

print("Сессии с информацией о выходных днях (первые 10 строк):")
sessions_with_weekend_df.show(10)


Сессии с информацией о выходных днях (первые 10 строк):


+--------------------+-------------------+-------------------+---------------------+----------+
|                puid|hours_sessions_long|msk_business_dt_str|minutes_sessions_long|is_weekend|
+--------------------+-------------------+-------------------+---------------------+----------+
|68296628-f9d6-11e...| 0.0377777777777777|         2024-11-26|                    2|     false|
|682966dc-f9d6-11e...|                0.0|         2024-11-26|                    0|     false|
|682966dc-f9d6-11e...|                0.0|         2024-11-26|                    0|     false|
|68296704-f9d6-11e...|                0.0|         2024-11-26|                    0|     false|
|68296722-f9d6-11e...|                0.0|         2024-11-26|                    0|     false|
|68296740-f9d6-11e...| 0.4890794444444444|         2024-11-26|                   29|     false|
|68296740-f9d6-11e...|          0.1847275|         2024-11-26|                   11|     false|
|6829675e-f9d6-11e...| 1.453055555555555

In [8]:
# Суммируем минуты для выходных и будних дней
weekend_weekday_stats_df = sessions_with_weekend_df.groupBy("is_weekend").agg(
    spark_sum("minutes_sessions_long").alias("total_minutes")
).orderBy("is_weekend")

print("Суммарные минуты для выходных и будних дней:")
weekend_weekday_stats_df.show()

Суммарные минуты для выходных и будних дней:


+----------+-------------+
|is_weekend|total_minutes|
+----------+-------------+
|     false|     17995153|
|      true|      6598993|
+----------+-------------+



In [10]:
# Анализ по возрастному рейтингу контента и дням недели
content_age_weekend_stats_df = audition_df.select(
    col("adult_content_flg"),
    col("hours_sessions_long"),
    col("msk_business_dt_str")
).withColumn(
    "minutes_sessions_long", 
    (col("hours_sessions_long") * 60).cast(IntegerType())
).withColumn(
    "is_weekend",
    when(dayofweek(to_date(col("msk_business_dt_str"), "yyyy-MM-dd")).isin([1, 7]), True).otherwise(False)
).groupBy("adult_content_flg", "is_weekend").agg(
    spark_sum("minutes_sessions_long").alias("total_minutes")
).filter(col("adult_content_flg").isNotNull() & col("is_weekend").isNotNull())

# Сортируем
content_age_weekend_stats_df = content_age_weekend_stats_df.orderBy("adult_content_flg", "is_weekend")

print("Анализ по взрослому/невзрослому контенту и выходным/будним дням:")
content_age_weekend_stats_df.show()

Анализ по взрослому/невзрослому контенту и выходным/будним дням:


+-----------------+----------+-------------+
|adult_content_flg|is_weekend|total_minutes|
+-----------------+----------+-------------+
|            false|     false|      3369978|
|            false|      true|      1401035|
|             true|     false|     14625175|
|             true|      true|      5197958|
+-----------------+----------+-------------+



**Вывод:**
- Как видно из вывода по анализ по возрастному рейтингу контента и дням недели наибольшую активность показывает взрослый контент в будниие дни а наименьшую детский в выходные однако можно заметить что длительность контента в выходные составляет примерно треть от контента в будни в раках одной категории что в среднем значении на сутки даст примерно одинаковый результат (предворительно, для большей точности требуется отдельный анализ) 

## Шаг 3. Соединение таблиц

In [11]:
# Объединяем таблицы по main_content_id
joined_audition_content_df = audition_df.join(content_df, "main_content_id", "inner")

print(f"Количество строк после объединения: {joined_audition_content_df.count()}")


Количество строк после объединения: 996565


In [12]:
# Удаляем лишние столбцы
cleaned_joined_df = joined_audition_content_df.drop(
    "main_author_name", 
    "app_version", 
    "usage_geo_id_name", 
    "usage_country_name"
)

print("Объединенная таблица без лишних столбцов (первые 10 строк):")
cleaned_joined_df.show(10)

Объединенная таблица без лишних столбцов (первые 10 строк):


+---------------+-----------+--------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+-----------------+--------------------+---------------------------+--------------------------+
|main_content_id|audition_id|                puid|usage_platform_ru|msk_business_dt_str|adult_content_flg|             hours|hours_sessions_long|kids_content_flg|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|
+---------------+-----------+--------------------+-----------------+-------------------+-----------------+------------------+-------------------+----------------+-----------------+--------------------+---------------------------+--------------------------+
|       oCURrBKV|          0|68296628-f9d6-11e...|          Станция|         2024-11-26|            false|0.0377777777777777| 0.0377777777777777|            true|        Audiobook|Фиксики. Интернет...|                  0.8302778|

In [13]:
# Считаем количество уникальных пользователей
unique_users_joined = cleaned_joined_df.select("puid").distinct().count()
unique_users_original = audition_df.select("puid").distinct().count()

print(f"\nКоличество уникальных пользователей в объединенной таблице: {unique_users_joined}")
print(f"Количество уникальных пользователей в исходной audition_df: {unique_users_original}")


Количество уникальных пользователей в объединенной таблице: 31010
Количество уникальных пользователей в исходной audition_df: 31063


**Обьяснение разницы**
- Разница возникает из-за использования INNER JOIN - в объединенную таблицу попадают только те записи, для которых есть соответствие в обеих таблицах по main_content_id. Пользователи, чьи сессии относятся к контенту, отсутствующему в справочнике content, исключаются.

In [14]:
# Выводим уникальные значения main_content_type
unique_content_types = cleaned_joined_df.select("main_content_type").distinct().collect()
unique_types_list = [row.main_content_type for row in unique_content_types]

print(f"Уникальные значения main_content_type: {unique_types_list}")

Уникальные значения main_content_type: ['Audiobook', 'Book', 'Comicbook']
